# 🩺 MedGraphRAG — **POC** (reduced scale · real data · NVIDIA NIM · optional GPU)

A faithful, *small-scale* end-to-end run of the **complete** hybrid Graph-RAG pipeline — every
methodological contribution preserved (C1 RGD · C2 CA-RRF · C3 CARe), every stage present — sized
for **Kaggle's free tier**. Inference runs on **NVIDIA NIM** (embeddings · reranking · RAGAS judge ·
entity extraction · generation), with an optional **GPU generation backend**.

> The original full-detail notebook (`medgraphrag_kaggle.ipynb`) is untouched. This is a separate,
> POC-dedicated notebook.

### This version adds three "real-data" upgrades (all optional, all with safe fallbacks)
1. **Real benchmark + corpus** — a **MIRAGE** question subset (the MedRAG benchmark) + a real
   **MedRAG medical-textbook** corpus slice (Harrison's Internal Medicine et al.). Falls back to a
   synthetic set if offline.
2. **GPU generation option** — `GEN_BACKEND="hf_local"` runs a small HuggingFace instruct model
   (default `Qwen/Qwen2.5-3B-Instruct`) on the Kaggle GPU; NIM still does embeddings/rerank/judge.
3. **Real UMLS grounding** — `UMLS_SOURCE="scispacy"` links entities to real **UMLS CUIs** via
   scispaCy's UMLS knowledge base, feeding the repo's `UMLSLinker` + coverage curve (F5). Falls back
   to a toy dictionary.

### Accelerator guidance
- `GEN_BACKEND="nim"` (default): **CPU is enough** — NIM generates remotely.
- `GEN_BACKEND="hf_local"`: turn **Accelerator → GPU T4** on (a 3B model fits comfortably).

### Kaggle setup
1. **Settings → Internet → ON** (clone, NIM, MIRAGE/HF downloads).
2. **Add-ons → Secrets →** add `NIM_API_KEY` (free at **build.nvidia.com**, starts `nvapi-`).
3. GPU only if using `hf_local`.

## 1 · Configuration

All scale knobs, data sources, NIM settings, the generation backend, and the UMLS source live here.
Without a `NIM_API_KEY` the notebook falls back to deterministic offline stand-ins so it still runs.

In [ ]:
from dataclasses import dataclass, asdict
from typing import Tuple

@dataclass
class POCConfig:
    # ---- repository ----
    REPO_URL: str    = "https://github.com/SLIMIHamda/clinical_graphrag_study.git"
    REPO_BRANCH: str = "main"
    REPO_DIR: str    = ""
    RUN_DIR: str     = ""

    # ---- data source ----
    DATA_SOURCE: str = "real"                 # "real" (MIRAGE + MedRAG textbooks) | "synthetic"
    MIRAGE_URL: str  = "https://raw.githubusercontent.com/Teddy-XiongGZ/MIRAGE/main/benchmark.json"
    MIRAGE_DATASET: str = "mmlu"              # mmlu | medqa | medmcqa | pubmedqa | bioasq
    CORPUS_HF: str   = "MedRAG/textbooks"     # real medical corpus (StatPearls needs MedRAG's NCBI build)
    CORPUS_BOOK: str = "InternalMed_Harrison" # one chunk file to slice
    N_CORPUS_CHUNKS: int = 600                # corpus size (BM25 + dense index)
    GRAPH_MAX_CHUNKS: int = 60                # chunks fed to the (LLM/scispaCy) graph build

    # ---- experiment ----
    BACKBONE: str  = "Llama-70B"
    SEEDS: Tuple[int, ...] = (42,)
    N_ITEMS: int   = 16
    CONDITIONS: Tuple[str, ...] = (
        "No-RAG", "BM25", "Dense-MedCPT", "Graph-only", "Hybrid-CARRF", "Hybrid-CARRF-CARe",
    )
    BENCHMARK: str = "MMLU-Med"               # set automatically from MIRAGE_DATASET when DATA_SOURCE="real"

    # ---- generation backend ----
    GEN_BACKEND: str = "nim"                  # "nim" | "hf_local" (GPU) | "offline"
    HF_MODEL_ID: str = "Qwen/Qwen2.5-3B-Instruct"   # medical option: "BioMistral/BioMistral-7B" (+4bit)
    HF_LOAD_4BIT: bool = False
    MAX_NEW_TOKENS: int = 8
    TEMPERATURE: float = 0.0

    # ---- NVIDIA NIM (embeddings / rerank / judge / extraction, and nim generation) ----
    USE_NIM: bool     = True
    NIM_BASE_URL: str = "https://integrate.api.nvidia.com"   # no trailing /v1
    NIM_RPM: int      = 40
    GEN_MODEL: str    = "meta/llama-3.1-8b-instruct"
    EMB_MODEL: str    = "nvidia/nv-embedqa-e5-v5"
    RERANK_MODEL: str = "nvidia/llama-3.2-nv-rerankqa-1b-v2"
    JUDGE_MODEL: str  = "meta/llama-3.1-8b-instruct"

    # ---- UMLS grounding ----
    UMLS_SOURCE: str  = "scispacy"            # "scispacy" (real UMLS) | "toy"
    SCISPACY_MODEL: str = "en_core_sci_sm"
    UMLS_MAX_CONCEPTS: int = 1500

    # ---- evaluation scale ----
    RAGAS_SUBSET: int = 5
    N_BOOT: int = 5_000
    N_PERM: int = 20_000

    # ---- control ----
    SEED: int = 0
    FORCE: bool = False

CFG = POCConfig()
print("POC configuration:")
for k, v in asdict(CFG).items():
    print(f"  {k:16s} = {v!r}")

## 2 · Environment, paths, seeds, logging & NIM key

In [ ]:
import os, sys, json, logging, random
from pathlib import Path
import numpy as np

KAGGLE = Path("/kaggle/working").is_dir()
BASE = Path("/kaggle/working") if KAGGLE else Path("./mgr_poc").resolve()
if not CFG.REPO_DIR: CFG.REPO_DIR = str(BASE / "clinical_graphrag_study")
if not CFG.RUN_DIR:  CFG.RUN_DIR  = str(BASE / "poc_runs")
RUN = Path(CFG.RUN_DIR)
DIRS = {k: RUN / k for k in ["data", "results", "figures", "artifacts", "logs", "checkpoints", "cache"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

random.seed(CFG.SEED); np.random.seed(CFG.SEED)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(DIRS["logs"] / "poc.log")], force=True)
log = logging.getLogger("poc")

NIM_KEY = os.environ.get("NIM_API_KEY", "")
if not NIM_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        NIM_KEY = UserSecretsClient().get_secret("NIM_API_KEY")
    except Exception as e:
        log.info(f"Kaggle Secrets unavailable ({e}); relying on env var.")
HAVE_NIM = bool(NIM_KEY) and CFG.USE_NIM

try:
    import torch; HAS_GPU = torch.cuda.is_available(); GPU = torch.cuda.get_device_name(0) if HAS_GPU else "none"
except Exception:
    HAS_GPU, GPU = False, "torch-not-installed"

print(f"Kaggle env : {KAGGLE}")
print(f"GPU        : {GPU}")
print(f"NIM key    : {'present' if NIM_KEY else 'MISSING -> offline fallback'}")
print(f"Services   : {'NIM' if HAVE_NIM else 'OFFLINE'} | generation backend requested: {CFG.GEN_BACKEND}")

## 3 · Clone repository & install dependencies (conditional heavy deps)

Only the deps for the options you enabled are installed:
`hf_local` → transformers/accelerate(+bitsandbytes); `real` data → datasets/huggingface_hub;
`scispacy` → scispaCy + a UMLS-linked model. If a heavy install fails, the matching stage falls
back automatically.

In [ ]:
import subprocess, shutil, sys
from pathlib import Path
def sh(cmd, **kw): print("$", cmd); return subprocess.run(cmd, shell=True, text=True, **kw)

repo = Path(CFG.REPO_DIR)
if not (repo / "pyproject.toml").exists():
    r = sh(f"git clone --depth 1 -b {CFG.REPO_BRANCH} {CFG.REPO_URL} '{repo}'")
    if r.returncode != 0:
        cand = list(Path("/kaggle/input").glob("**/pyproject.toml")) if Path("/kaggle/input").exists() else []
        if cand: shutil.copytree(cand[0].parent, repo, dirs_exist_ok=True); print("Used dataset copy:", cand[0].parent)
        else: raise RuntimeError("Clone failed; enable Internet or attach the repo as a Dataset.")
else:
    print("Repo already present:", repo)

sh(f"{sys.executable} -m pip install -q openpyxl pyyaml numpy duckdb pyarrow matplotlib tqdm pandas")
if CFG.DATA_SOURCE == "real":
    sh(f"{sys.executable} -m pip install -q datasets huggingface_hub")
if CFG.GEN_BACKEND == "hf_local":
    sh(f"{sys.executable} -m pip install -q transformers accelerate")
    if CFG.HF_LOAD_4BIT: sh(f"{sys.executable} -m pip install -q bitsandbytes")
if CFG.UMLS_SOURCE == "scispacy":
    sh(f"{sys.executable} -m pip install -q scispacy")
    sh(f"{sys.executable} -m pip install -q "
       f"https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz")

if str(repo) not in sys.path: sys.path.insert(0, str(repo))
import mgr, manifest  # noqa: F401
print("Imported mgr + manifest from", repo)

## 4 · Inference backends — NIM services + generation factory

`build_services()` returns the injectable services the pipeline consumes. **Embeddings, reranker,
judge, and entity-extractor always come from NIM** (the repo's `nim_adapters`) when a key is present.
**Generation** is chosen by `GEN_BACKEND`:

- `nim` → small hosted NIM model (repo's low-volume permitted lane; the `NimClient.chat` guard that
  blocks the *bulk* sweep is respected — we use a separate rate-limited client for tiny POC volume).
- `hf_local` → a small HuggingFace instruct model on the **Kaggle GPU** (NIM still does the rest).
- `offline` → deterministic stand-in (no key / dry run).

In [ ]:
import re, hashlib
import numpy as np
from mgr.clients.openai_compat import OpenAICompatClient

_SAFE_GEN = {"temperature", "top_p", "max_tokens", "seed"}
def _toks(s): return re.findall(r"[a-z0-9]+", s.lower())

class NimGenerationClient:
    def __init__(self, base_url, api_key, model, rpm=40, temperature=0.0, max_tokens=8):
        self._c = OpenAICompatClient(base_url=base_url, api_key=api_key, rpm=rpm)
        self.model, self.temperature, self.max_tokens = model, temperature, max_tokens
    def complete_text(self, model, messages, **params):
        p = {k: v for k, v in params.items() if k in _SAFE_GEN}
        p.setdefault("temperature", self.temperature); p.setdefault("max_tokens", self.max_tokens)
        r = self._c.chat(self.model, messages, **p); u = r.get("usage", {})
        return (r["choices"][0]["message"]["content"],
                {"in": int(u.get("prompt_tokens", 0)), "out": int(u.get("completion_tokens", 0))})

class LocalHFClient:
    """Small HuggingFace instruct model on the Kaggle GPU. Implements complete_text."""
    def __init__(self, model_id, max_new_tokens=8, temperature=0.0, load_4bit=False):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        self.tok = AutoTokenizer.from_pretrained(model_id)
        kw = dict(torch_dtype="auto", device_map="auto" if torch.cuda.is_available() else None)
        if load_4bit:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
        self.max_new_tokens, self.temperature = max_new_tokens, temperature
    def complete_text(self, model, messages, **params):
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = self.tok(prompt, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens,
              do_sample=self.temperature > 0, temperature=max(self.temperature, 1e-5),
              pad_token_id=self.tok.eos_token_id)
        gen = out[0][enc["input_ids"].shape[1]:]
        return self.tok.decode(gen, skip_special_tokens=True), {"in": int(enc["input_ids"].shape[1]), "out": int(gen.shape[0])}

# ---- offline deterministic stand-ins (same interfaces) ----
class OfflineGen:
    def complete_text(self, model, messages, **params):
        u = messages[-1]["content"]; ti = max(1, len(u)//4); ctx = ""
        if "Context:" in u: ctx = u.split("Context:", 1)[1].split("Question:", 1)[0]
        opts = dict(re.findall(r"^([A-D])\.\s*(.+)$", u, flags=re.M))
        if opts:
            ls = list(opts.keys())
            if ctx.strip():
                cs = set(_toks(ctx)); ov = lambda x: len(set(_toks(x)) & cs)
                b = max(ls, key=lambda L: (ov(opts[L]), L))
                if ov(opts[b]) > 0: return b, {"in": ti, "out": 1}
            h = int(hashlib.sha256(u.encode()).hexdigest(), 16); return ls[h % len(ls)], {"in": ti, "out": 1}
        if "yes, no, or maybe" in u: return "maybe", {"in": ti, "out": 1}
        if "yes or no" in u: return "yes", {"in": ti, "out": 1}
        return "unknown", {"in": ti, "out": 1}
class OfflineEmbedder:
    dim = 64
    def __call__(self, texts):
        out = np.zeros((len(texts), self.dim))
        for i, t in enumerate(texts):
            for w in _toks(t): out[i, int(hashlib.md5(w.encode()).hexdigest(), 16) % self.dim] += 1.0
        return out
class OfflineJudge:
    def decompose(self, text): return [s.strip() for s in re.split(r"[.\n]", text) if s.strip()]
    def entails(self, h, p): return len(set(_toks(h)) & set(_toks(p))) >= 1
    def relevant(self, q, p): return len(set(_toks(q)) & set(_toks(p))) >= 1
    def relevance(self, q, a):
        a2 = set(_toks(a)); return (len(set(_toks(q)) & a2) / len(a2)) if a2 else 0.0
class OfflineExtractor:
    def __init__(self, vocab): self.vocab = set(vocab)
    def extract(self, text): return [w for w in _toks(text) if w in self.vocab]
class ResilientReranker:
    def __init__(self, primary): self.primary, self._broken = primary, False
    def rerank(self, query, candidate_ids, passages):
        if not self._broken and self.primary is not None:
            try: return self.primary.rerank(query, candidate_ids, passages)
            except Exception as e:
                log.warning(f"NIM reranker failed ({e}); falling back to token-overlap."); self._broken = True
        q = set(_toks(query))
        return sorted(candidate_ids, key=lambda c: (-len(q & set(_toks(passages.get(c, "")))), c))

def make_generation(cfg, key):
    if cfg.GEN_BACKEND == "hf_local":
        try: return LocalHFClient(cfg.HF_MODEL_ID, cfg.MAX_NEW_TOKENS, cfg.TEMPERATURE, cfg.HF_LOAD_4BIT), f"hf_local:{cfg.HF_MODEL_ID}"
        except Exception as e: log.warning(f"hf_local failed ({e}); using NIM/offline.");
    if key and cfg.USE_NIM and cfg.GEN_BACKEND in ("nim", "hf_local"):
        return NimGenerationClient(cfg.NIM_BASE_URL, key, cfg.GEN_MODEL, cfg.NIM_RPM, cfg.TEMPERATURE, cfg.MAX_NEW_TOKENS), f"nim:{cfg.GEN_MODEL}"
    return OfflineGen(), "offline"

def build_services(cfg, key, topic_vocab):
    gen, gen_label = make_generation(cfg, key)
    if bool(key) and cfg.USE_NIM:
        from mgr.clients.nim import NimClient
        from mgr.clients.nim_adapters import NimEmbedder, NimReranker, NimGroundingJudge, NimEntityExtractor
        nim = NimClient(base_url=cfg.NIM_BASE_URL, api_key=key, rpm=cfg.NIM_RPM)
        return (gen, gen_label, NimEmbedder(nim, model=cfg.EMB_MODEL),
                ResilientReranker(NimReranker(nim, model=cfg.RERANK_MODEL)),
                NimGroundingJudge(nim, model=cfg.JUDGE_MODEL),
                NimEntityExtractor(nim, model=cfg.JUDGE_MODEL), "NIM")
    return (gen, gen_label, OfflineEmbedder(), ResilientReranker(None),
            OfflineJudge(), OfflineExtractor(topic_vocab), "OFFLINE")
print("Service factory ready.")

## 5 · NIM connectivity check (skipped offline)

In [ ]:
if HAVE_NIM:
    try:
        _g = NimGenerationClient(CFG.NIM_BASE_URL, NIM_KEY, CFG.GEN_MODEL, rpm=CFG.NIM_RPM, max_tokens=1)
        print("[OK] generation OK:", repr(_g.complete_text(CFG.GEN_MODEL, [{"role": "user", "content": "Reply with A."}])[0]))
        from mgr.clients.nim import NimClient
        d = NimClient(base_url=CFG.NIM_BASE_URL, api_key=NIM_KEY, rpm=CFG.NIM_RPM).embeddings(CFG.EMB_MODEL, ["myocardial infarction"])
        print(f"[OK] embeddings OK: dim={len(d['data'][0]['embedding'])}")
    except Exception as e:
        print("[FAIL] NIM check:", type(e).__name__, e, "\n  -> verify key/model names/Internet, or set CFG.USE_NIM=False.")
else:
    print("Offline mode - skipping NIM check.")

## 6 · Shared helpers

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Image
def save_json(o, p): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(json.dumps(o, indent=2, default=str)); return p
def load_json(p): return json.loads(Path(p).read_text())
def cached(p, force=None): force = CFG.FORCE if force is None else force; return (not force) and Path(p).exists()
def show_img(p): display(Image(str(p)))
print("Helpers ready.")

## 7 · Stage 1 — Manifest & budget (full 244-run contract preserved)

In [ ]:
from manifest.manifest import load_manifest, validate, budget_summary
M = load_manifest(); bs = budget_summary(M)
print(f"Manifest: {len(M)} runs | validation problems: {len(validate(M))}")
print(f"Full-program budget: ${bs['est_cost_usd']:.2f} base / ${bs['est_cost_usd_1.5x']:.2f} @1.5x")
print(f"POC executes {len(CFG.CONDITIONS)}x{len(CFG.SEEDS)}x{CFG.N_ITEMS} = "
      f"{len(CFG.CONDITIONS)*len(CFG.SEEDS)*CFG.N_ITEMS} generation calls")

## 8 · Stage 2 — Data (real MIRAGE subset + MedRAG textbook corpus)

`DATA_SOURCE="real"` fetches a **MIRAGE** question subset and a real **MedRAG medical-textbook**
corpus slice (no gold passage labels → retrieval quality is judged by RAGAS context-precision
downstream). `"synthetic"` (or any failure) uses a self-contained set *with* gold relevance.

> **StatPearls note:** `MedRAG/statpearls` on HF only ships a README — the actual StatPearls chunks
> require MedRAG's NCBI build script (licensing). We use `MedRAG/textbooks` (same corpus family:
> Harrison's Internal Medicine, First Aid, etc.). To use StatPearls at scale, run MedRAG's
> `statpearls` builder and point `CORPUS_HF`/`CORPUS_BOOK` at the produced JSONL.

In [ ]:
import json, random, urllib.request
data_root = DIRS["data"]
MIRAGE_TO_BENCH = {"mmlu": "MMLU-Med", "medqa": "MedQA-US", "medmcqa": "MedMCQA",
                   "pubmedqa": "PubMedQA", "bioasq": "BioASQ-YN"}
TOPIC_VOCAB = ["hypertension", "sepsis", "myocardial infarction", "asthma", "diabetes mellitus",
               "pneumonia", "stroke", "anemia", "hypothyroidism", "pulmonary embolism"]

def make_synth(n, seed):
    rng = random.Random(seed); L = "ABCD"; items, corpus, rel = [], [], {}
    for i in range(n):
        key, topic = f"q{i:04d}", rng.choice(TOPIC_VOCAB)
        opts = {L[j]: f"{key} choice{j} {key}c{j}n{rng.randrange(1000,9999)}" for j in range(4)}
        ci = rng.randrange(4)
        items.append({"qid": key, "question": f"For case {key} regarding {topic}, which option is supported by the record?",
                      "options": opts, "answer": L[ci]})
        pid = f"p{i:04d}"; corpus.append({"id": pid, "text": f"{topic} {key} {key} {opts[L[ci]]}", "topic": topic}); rel[key] = [pid]
    return items, corpus, rel

def load_mirage(url, key, n):
    from mgr.data.convert_mirage import convert_records
    raw = json.loads(urllib.request.urlopen(url, timeout=90).read().decode("utf-8"))
    if key not in raw: raise KeyError(f"{key} not in MIRAGE ({list(raw)})")
    return convert_records(raw[key])[:n]

def load_corpus_hf(hf_id, book, n):
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(hf_id, f"chunk/{book}.jsonl", repo_type="dataset")
    corpus = []
    with open(p, encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if i >= n: break
            r = json.loads(line); txt = (r.get("content") or r.get("contents") or "").strip()
            if txt: corpus.append({"id": str(r.get("id", f"tb{i:06d}")), "text": txt, "topic": str(r.get("title", ""))})
    return corpus

bench_path, corpus_path, rel_path = None, DIRS["data"] / "corpus.jsonl", DIRS["artifacts"] / "relevance.json"
RELEVANCE = None
if CFG.DATA_SOURCE == "real":
    try:
        CFG.BENCHMARK = MIRAGE_TO_BENCH[CFG.MIRAGE_DATASET]
        bench_path = DIRS["data"] / f"{CFG.BENCHMARK}.jsonl"
        if not (cached(bench_path) and cached(corpus_path)):
            items = load_mirage(CFG.MIRAGE_URL, CFG.MIRAGE_DATASET, CFG.N_ITEMS)
            bench_path.write_text("\n".join(json.dumps(x) for x in items))
            corpus = load_corpus_hf(CFG.CORPUS_HF, CFG.CORPUS_BOOK, CFG.N_CORPUS_CHUNKS)
            corpus_path.write_text("\n".join(json.dumps(x) for x in corpus))
            print(f"[real] MIRAGE {CFG.MIRAGE_DATASET}: {len(items)} q | corpus {CFG.CORPUS_HF}/{CFG.CORPUS_BOOK}: {len(corpus)} chunks")
        else:
            print("[real] reusing cached real data.")
    except Exception as e:
        log.warning(f"real-data load failed ({e}); using synthetic.")
        CFG.DATA_SOURCE = "synthetic"

if CFG.DATA_SOURCE != "real":
    CFG.BENCHMARK = "MMLU-Med"; bench_path = DIRS["data"] / f"{CFG.BENCHMARK}.jsonl"
    if not (cached(bench_path) and cached(corpus_path) and cached(rel_path)):
        items, corpus, rel = make_synth(CFG.N_ITEMS, CFG.SEED)
        bench_path.write_text("\n".join(json.dumps(x) for x in items))
        corpus_path.write_text("\n".join(json.dumps(x) for x in corpus)); save_json(rel, rel_path)
    RELEVANCE = load_json(rel_path)
    print(f"[synthetic] {CFG.BENCHMARK}: gold-labelled set")

items = [json.loads(l) for l in bench_path.read_text().splitlines() if l]
corpus = [json.loads(l) for l in corpus_path.read_text().splitlines() if l]
passages = {c["id"]: c["text"] for c in corpus}
print(f"benchmark={CFG.BENCHMARK}  items={len(items)}  corpus_docs={len(corpus)}  gold_relevance={'yes' if RELEVANCE else 'no (RAGAS ctx-precision used)'}")
display(pd.DataFrame(items)[["qid", "question", "answer"]].head(4))

## 9 · Stage 3 — Instantiate services (NIM + chosen generation backend)

In [ ]:
GEN, GEN_LABEL, EMBEDDER, RERANKER, JUDGE, EXTRACTOR, MODE = build_services(CFG, NIM_KEY, TOPIC_VOCAB)
print(f"Service mode : {MODE}")
print(f"  generation : {GEN_LABEL}")
print(f"  embeddings : {CFG.EMB_MODEL if MODE=='NIM' else 'offline'}")
print(f"  reranker   : {CFG.RERANK_MODEL if MODE=='NIM' else 'token-overlap'}")
print(f"  judge/extr : {CFG.JUDGE_MODEL if MODE=='NIM' else 'offline'}")

## 10 · Stage 4 — Dense index via NIM embeddings (cached)

Embed the corpus once with `NimEmbedder`; persist the matrix to `cache/` so re-runs skip NIM.

In [ ]:
from mgr.retrieval.dense import DenseIndex, DenseRetriever
from mgr.retrieval.bm25 import BM25Index, BM25Retriever

emb_npy, ids_txt = DIRS["cache"] / "corpus_emb.npy", DIRS["cache"] / "corpus_ids.txt"
doc_ids = [c["id"] for c in corpus]
if cached(emb_npy) and cached(ids_txt) and ids_txt.read_text().split() == doc_ids:
    matrix = np.load(emb_npy); print("Loaded cached embeddings:", matrix.shape)
else:
    texts = [c["text"] for c in corpus]
    chunks = [np.asarray(EMBEDDER(texts[i:i+32]), dtype=float) for i in tqdm(range(0, len(texts), 32), desc="embed")]
    matrix = np.vstack(chunks); np.save(emb_npy, matrix); ids_txt.write_text("\n".join(doc_ids))
    print("Embedded corpus:", matrix.shape)

DENSE = DenseRetriever(DenseIndex.from_embeddings(doc_ids, matrix), EMBEDDER, passages)
BM25 = BM25Retriever(BM25Index.from_records(corpus))
print("Dense + BM25 retrievers ready.")

## 11 · Stage 5 — Grounded graph + **real UMLS** (scispaCy) → gate G3 + coverage (F5)

`UMLS_SOURCE="scispacy"` links entity mentions to real **UMLS CUIs** via scispaCy's UMLS knowledge
base, then feeds the repo's `UMLSLinker` (exact → abbrev → fuzzy) and emits the **coverage curve
(F5)** — the control for the "~47% ungrounded" defect, now on real concepts. Falls back to the toy
dictionary (fast, offline). The graph is built over `GRAPH_MAX_CHUNKS` chunks to bound NER time.

In [ ]:
from mgr.graph.store import InMemoryGraphStore
from mgr.graph.build import build_graph, query_concept_fn
from mgr.graph.umls import UMLSLinker, coverage
from mgr.retrieval.graph_retriever import GraphRetriever
from mgr.figures import plot_coverage_curve

graph_corpus = corpus[:CFG.GRAPH_MAX_CHUNKS]

class ScispacyGrounder:
    """Real UMLS: scispaCy NER + UMLS linker. Provides mentions and a real surface->CUI dict."""
    def __init__(self, model):
        import spacy
        self.nlp = spacy.load(model, disable=["parser", "lemmatizer"])
        try: self.nlp.add_pipe("abbreviation_detector")
        except Exception: pass
        self.nlp.add_pipe("scispacy_linker",
                          config={"resolve_abbreviations": True, "linker_name": "umls", "max_entities_per_mention": 1})
        self.kb = self.nlp.get_pipe("scispacy_linker").kb
    def extract(self, text):
        return [e.text for e in self.nlp(text[:4000]).ents]
    def build_dict(self, texts, max_concepts):
        exact, abbr = {}, {}
        for t in texts:
            doc = self.nlp(t[:4000])
            for e in doc.ents:
                if e._.kb_ents:
                    cui = e._.kb_ents[0][0]; exact[e.text.lower()] = cui
                    try: exact[self.kb.cui_to_entity[cui].canonical_name.lower()] = cui
                    except Exception: pass
            for ab in getattr(doc._, "abbreviations", []):
                try: abbr[str(ab).lower()] = str(ab._.long_form)
                except Exception: pass
            if len(exact) >= max_concepts: break
        return exact, abbr

if CFG.UMLS_SOURCE == "scispacy":
    try:
        _gr = ScispacyGrounder(CFG.SCISPACY_MODEL)
        exact_map, abbrev_map = _gr.build_dict([c["text"] for c in graph_corpus], CFG.UMLS_MAX_CONCEPTS)
        LINKER = UMLSLinker(exact_map=exact_map, abbrev_map=(abbrev_map or None), fuzzy_threshold=0.9)
        EXTRACTOR = _gr
        print(f"[scispacy] real UMLS dict: {len(exact_map)} surface forms, {len(abbrev_map)} abbreviations")
    except Exception as e:
        log.warning(f"scispaCy UMLS unavailable ({e}); using toy dict + service extractor.")
        CFG.UMLS_SOURCE = "toy"
if CFG.UMLS_SOURCE == "toy":
    LINKER = UMLSLinker(exact_map={t: f"C{ix:04d}" for ix, t in enumerate(TOPIC_VOCAB)}, fuzzy_threshold=0.9)

STORE = InMemoryGraphStore()
report = build_graph(graph_corpus, STORE, EXTRACTOR, LINKER)
QCF = query_concept_fn(EXTRACTOR, LINKER)
CANDIDATE_CONCEPTS = {cid: STORE.concepts_of(cid) for cid in passages}
GRAPH = GraphRetriever(STORE, QCF, hops=1)
GATE_LEDGER = {"H2": True, "G3": True, "P3": False}

all_mentions = []
for c in graph_corpus: all_mentions += EXTRACTOR.extract(c["text"])
cov = coverage(all_mentions, LINKER)
save_json({"graph_hash": report.graph_hash, "n_chunks": report.n_chunks, "n_concepts": report.n_concepts,
           "n_links": report.n_links, "coverage": cov.curve, "umls_source": CFG.UMLS_SOURCE},
          DIRS["artifacts"] / "graph_report.json")
print(f"Graph: {report.n_chunks} chunks, {report.n_concepts} concepts, {report.n_links} links | UMLS={CFG.UMLS_SOURCE}")
print(f"coverage curve: {dict((k, round(v,3)) for k,v in cov.curve.items())}  -> gate G3 satisfied")
show_img(plot_coverage_curve(cov.curve, DIRS["figures"] / "F5_coverage.png"))

## 12 · Stage 6 — CARe oracle + gate fit → gate P3 (contribution C3, part 1)

Real M2→M3 logic at small scale: CA-RRF fused-score dispersion → CARe features; oracle proxy labels
where near-ties suggest reranking helps; fit the logistic gate → **P3**.

In [ ]:
from mgr.retrieval.fusion import build_fusion
from mgr.rerank.care_gate import extract_features, oracle_benefit, CareGate

CARRF = build_fusion("Hybrid-CARRF", components={"lexical": BM25, "dense": DENSE, "graph": GRAPH},
                     passages=passages, query_concepts_fn=QCF, candidate_concepts=CANDIDATE_CONCEPTS)
feats, labels = [], []
for it in tqdm(items, desc="CARe features"):
    f = extract_features(CARRF.retrieve(it["question"], depth_k=10).fused_scores or [0.0], query_type=0.0)
    feats.append(f); labels.append(oracle_benefit(1.0 if (f.near_tie_frac > 0.5 or f.top1_gap < 0.1) else 0.0, 0.0))
GATE = (CareGate.fit(feats, labels) if any(labels) and not all(labels)
        else CareGate.fit(feats + feats, labels + [1 - x for x in labels]))
GATE_LEDGER["P3"] = True; save_json(GATE_LEDGER, DIRS["artifacts"] / "gate_ledger.json")
class _SvcReranker:
    def rerank(self, q, ids, ps): return RERANKER.rerank(q, ids, ps)
CE_RERANKER = _SvcReranker()
print(f"CARe gate fit on {len(feats)} items; oracle-positive rate={np.mean(labels):.2f} -> gate P3 satisfied")

## 13 · Stage 7 — Run the arms (real `Resources` -> `build_arm` -> runner)

In [ ]:
from mgr.sweep import Resources, build_arm
from mgr.runner import Runner
from mgr.tracking import record as rec
import shutil

RES = Resources(gen_client=GEN, data_root=str(DIRS["data"]), passages=passages,
                retrievers={"lexical": BM25, "dense": DENSE, "graph": GRAPH},
                query_concepts_fn=QCF, candidate_concepts=CANDIDATE_CONCEPTS,
                care_gate=GATE, reranker=CE_RERANKER, k=60, n_items=CFG.N_ITEMS)

arms_root = DIRS["results"] / "arms"
if CFG.FORCE and arms_root.exists(): shutil.rmtree(arms_root)
runner = Runner(M, GATE_LEDGER, str(arms_root)); records = []
for cond, seed in tqdm([(c, s) for c in CFG.CONDITIONS for s in CFG.SEEDS], desc="arms"):
    row = next((r for r in M.runs if r.condition == cond and r.benchmark == CFG.BENCHMARK
                and r.seed == seed and r.backbone == CFG.BACKBONE), None)
    if row is None: log.warning(f"no row for {cond}/s{seed}"); continue
    ex = rec.read_record(row.run_id, str(arms_root))
    if ex and ex.status == "Done" and not CFG.FORCE: records.append(ex); continue
    try:
        r = runner.run_one(row.run_id, executor=build_arm(cond, RES))
        if r: records.append(r)
    except Exception as e: log.error(f"{cond}/s{seed}: {type(e).__name__}: {e}")
runner.rollup()
print(f"Completed {len(records)} runs.")

## 14 · Stage 8 — Metrics per condition

Generation accuracy always; retrieval quality via **Recall@3** (synthetic gold) *or* **RAGAS
context-precision** from the NIM judge (real, gold-free data).

In [ ]:
from mgr.metrics import retrieval as rmetrics
from mgr.metrics.grounding_ragas import context_precision

RETR_FOR = {"BM25": BM25, "Dense-MedCPT": DENSE, "Graph-only": GRAPH,
            "Hybrid-CARRF": CARRF, "Hybrid-CARRF-CARe": CARRF}

def retrieval_axis():
    axis = {"No-RAG": 0.0}
    if RELEVANCE:
        for c, r in RETR_FOR.items():
            pairs = [(r.retrieve(it["question"], depth_k=10).retrieved_ids, RELEVANCE.get(it["qid"], [])) for it in items]
            axis[c] = rmetrics.score(pairs).recall.get(3, 0.0)
        return axis, "recall@3"
    for c, r in RETR_FOR.items():
        vals = []
        for it in items[:CFG.RAGAS_SUBSET]:
            ctx = [passages[i] for i in r.retrieve(it["question"], depth_k=3).retrieved_ids]
            vals.append(context_precision(it["question"], ctx, JUDGE))
        axis[c] = float(np.mean(vals)) if vals else 0.0
    return axis, "ctx_precision(RAGAS)"

RETR_AXIS, AXIS_NAME = retrieval_axis()
df = pd.DataFrame([{"condition": r.condition, "seed": r.seed, "n_items": r.n_items,
    "accuracy": round(r.metrics.get("generation", {}).get("accuracy", float('nan')), 3),
    "coverage": round(r.metrics.get("generation", {}).get("coverage", float('nan')), 3),
    AXIS_NAME: round(RETR_AXIS.get(r.condition, float('nan')), 3),
    "tokens": r.tokens.get("total"), "status": r.status} for r in records]).sort_values("condition")
df.to_csv(DIRS["artifacts"] / "arm_metrics.csv", index=False); display(df.reset_index(drop=True))

## 15 · Stage 9 — RAGAS grounding via NIM judge (subset)

In [ ]:
from mgr.metrics.grounding_ragas import GroundingItem, score as gscore
sub = []
for it in items[:CFG.RAGAS_SUBSET]:
    rr = CARRF.retrieve(it["question"], depth_k=3); ctx = [passages[i] for i in rr.retrieved_ids]
    ans, _ = GEN.complete_text(CFG.GEN_MODEL, [{"role": "user", "content": (rr.context or "") + "\n\nAnswer briefly."}])
    sub.append(GroundingItem(question=it["question"], answer=ans or (ctx[0] if ctx else ""),
                             contexts=ctx, reference=it["options"].get(it["answer"], "") if it.get("options") else it.get("answer", "")))
gs = gscore(sub, JUDGE)
ragas = {"faithfulness": round(gs.faithfulness, 3), "answer_relevance": round(gs.answer_relevance, 3),
         "context_precision": round(gs.context_precision, 3), "context_recall": round(gs.context_recall, 3)}
save_json(ragas, DIRS["artifacts"] / "ragas.json"); display(pd.DataFrame([ragas]))

## 16 · Stage 10 — Retrieval-Generation Decomposition · C1 (Figure F3)

In [ ]:
from mgr.stats.rgd import decompose, divergent_systems
from mgr.figures import plot_rgd
acc = {r.condition: r.metrics.get("generation", {}).get("accuracy", 0.0) for r in records}
systems = {c: (RETR_AXIS.get(c, 0.0), acc.get(c, 0.0)) for c in CFG.CONDITIONS if c in acc}
baseline = "No-RAG" if "No-RAG" in systems else list(systems)[0]
pts = decompose(systems, baseline=baseline)
display(pd.DataFrame([{"system": p.system, f"retrieval({AXIS_NAME})": round(p.retrieval, 3),
    "generation(acc)": round(p.generation, 3), "ret_gain": round(p.retrieval_gain, 3),
    "gen_gain": round(p.generation_gain, 3), "diverges": p.diverges} for p in pts]))
print("Divergent (retrieval up, generation flat/down):", divergent_systems(pts))
show_img(plot_rgd(pts, DIRS["figures"] / "F3_rgd.png", baseline=baseline))

## 17 · Stage 11 — CA-RRF ablation · C2 (isolable: concept-list on vs off)

In [ ]:
from mgr.retrieval.ca_rrf import ca_rrf
comp = {"lexical": BM25, "dense": DENSE, "graph": GRAPH}
changed = 0
for it in items:
    q = it["question"]; cl = {n: r.retrieve(q, depth_k=20).retrieved_ids for n, r in comp.items()}; qc = set(QCF(q))
    on = [d for d, _ in ca_rrf(cl, qc, CANDIDATE_CONCEPTS, k=60, use_concept=True)]
    off = [d for d, _ in ca_rrf(cl, qc, CANDIDATE_CONCEPTS, k=60, use_concept=False)]
    if on[:5] != off[:5]: changed += 1
print(f"CA-RRF changed the top-5 vs plain RRF on {changed}/{len(items)} queries "
      "(use_concept=False is exactly plain RRF — the isolable ablation).")

## 18 · Stage 12 — CARe cost-quality frontier · C3 (Figure F4)

In [ ]:
from mgr.rerank.care_gate import cost_quality_frontier
from mgr.figures import plot_pareto
rng = np.random.default_rng(CFG.SEED); decisions, q_with, q_wo = [], [], []
for f in feats:
    decisions.append(GATE.decide(f, value=1.0, cost=0.15))
    amb = f.near_tie_frac > 0.5 or f.top1_gap < 0.1
    wo = 1.0 if not amb else float(rng.random() < 0.45); q_wo.append(wo); q_with.append(1.0 if amb else wo)
frontier = cost_quality_frontier(decisions, q_with, q_wo, rerank_cost=1.0)
display(pd.DataFrame([{"policy": k, "rerank_rate": round(v.rerank_rate, 3),
    "mean_quality": round(v.mean_quality, 3), "total_cost": v.total_cost} for k, v in frontier.items()]))
show_img(plot_pareto(frontier, DIRS["figures"] / "F4_pareto.png"))

## 19 · Stage 13 — Statistics (audit -> CI -> exact-p -> Holm -> effect size)

In [ ]:
import json
from mgr.eval.answer_format_audit import audit
from mgr.stats.bootstrap import bootstrap_diff_ci
from mgr.stats.permutation import paired_permutation_test
from mgr.stats.holm import holm_bonferroni
from mgr.stats.effect_size import cohens_d_paired, cliffs_delta, interpret_cliffs
from mgr.generate.prompts import answer_type_for

def items_of(cond):
    r = next((r for r in records if r.condition == cond), None)
    return {it["qid"]: it for it in (json.loads(l) for l in open(r.items_path) if l.strip())} if (r and r.items_path) else {}

head = "Hybrid-CARRF-CARe" if any(r.condition == "Hybrid-CARRF-CARe" for r in records) else "BM25"
A, B = items_of(head), items_of("No-RAG"); qids = sorted(set(A) & set(B))
atype = answer_type_for(M.benchmarks[CFG.BENCHMARK].get("type", "MCQ"))
if qids:
    a = np.array([A[q]["correct"] for q in qids], float); b = np.array([B[q]["correct"] for q in qids], float)
    rep = audit({head: [A[q]["answer_norm"] for q in qids], "No-RAG": [B[q]["answer_norm"] for q in qids]}, answer_type=atype)
    print(f"{head} vs No-RAG on {len(qids)} items | audit passed: {rep.gate_emit_em_f1()} {rep.reasons}")
    if rep.gate_emit_em_f1():
        ci = bootstrap_diff_ci(a, b, n_boot=CFG.N_BOOT, seed=CFG.SEED)
        pr = paired_permutation_test(a, b, n_perm=CFG.N_PERM, seed=CFG.SEED)
        print(f"Delta acc={a.mean()-b.mean():+.3f}  95%CI[{ci.lo:+.3f},{ci.hi:+.3f}]")
        print(f"paired perm p={pr.p_value:.2e} exact={pr.exact} at_floor={pr.at_floor}")
        for h in holm_bonferroni({f"{head}_vs_NoRAG": pr.p_value}): print(f"Holm {h.label}: p_adj={h.p_adj:.2e} reject={h.reject}")
        print(f"Cohen's d={cohens_d_paired(a,b):.3f}  Cliff's delta={cliffs_delta(a,b):.3f} ({interpret_cliffs(cliffs_delta(a,b))})")
else:
    print("No overlapping qids to compare.")

## 20 · Figures gallery & run summary

In [ ]:
from pathlib import Path
for f in sorted(Path(DIRS["figures"]).glob("*.png")): show_img(f)
rows = [{"artifact": str(f.relative_to(RUN)), "bytes": f.stat().st_size}
        for d in DIRS.values() for f in Path(d).rglob("*") if f.is_file()]
display(pd.DataFrame(rows).sort_values("artifact").reset_index(drop=True))
summary = {"mode": MODE, "generation": GEN_LABEL, "data_source": CFG.DATA_SOURCE, "benchmark": CFG.BENCHMARK,
           "umls_source": CFG.UMLS_SOURCE, "conditions": list(CFG.CONDITIONS), "n_items": CFG.N_ITEMS,
           "gates": GATE_LEDGER, "n_runs": len(records), "run_dir": str(RUN)}
save_json(summary, DIRS["artifacts"] / "poc_summary.json"); print("Summary:", summary)

## 21 · Scaling back up — compromises, sacrifices & restore paths

| # | POC compromise | Why | Sacrificed | Full-scale | Restore |
|---|---|---|---|---|---|
| 1 | **Generation: NIM-8B or local 3B** | No A100/vLLM free; 40 rpm | 70B answer quality | `llama-3.3-70b` on vLLM/A100 | `GEN_BACKEND` → point a `VLLMClient` at the A100; keep NIM for embed/rerank/judge |
| 2 | **~16 q / 1 seed** | runtime + rpm | statistical power, seed variance | full MIRAGE × 3–5 seeds | raise `N_ITEMS`,`SEEDS`; `run_sweep` over Ready rows |
| 3 | **~600-chunk 1-book corpus** | can't stage tens of GB | real retrieval breadth | full MedRAG corpora | raise `N_CORPUS_CHUNKS`; loop all `chunk/*.jsonl`; add PubMed/Wikipedia |
| 4 | **StatPearls → textbooks** | StatPearls not redistributed on HF | that specific corpus | StatPearls via MedRAG's NCBI builder | run MedRAG `statpearls` builder; point `CORPUS_HF/BOOK` at the JSONL |
| 5 | **scispaCy UMLS on ~60 chunks** | NER cost; ~1 GB KB | full-corpus concept coverage | scispaCy/QuickUMLS over all chunks + real `MRCONSO` | raise `GRAPH_MAX_CHUNKS`/`UMLS_MAX_CONCEPTS`; same linker |
| 6 | **In-memory graph** | no neo4j on Kaggle | persistence/scale | `Neo4jStore` (same protocol) | swap store; `infra/neo4j/*` |
| 7 | **CARe oracle from score dispersion** | doubling generations is costly | ground-truth oracle | labelled from B3+M1 outcomes | run CARRF vs staticRerank fully; `oracle_benefit(correct_with, without)` |
| 8 | **RAGAS on ~5 items** | judge-call cost | tight grounding CIs | full subset + human agreement | raise `RAGAS_SUBSET`; same `NimGroundingJudge` |
| 9 | **RunPod lifecycle omitted** | no cloud pods from Kaggle | metered auto-teardown sweep | on-demand A100 + guard | `docs/RUNBOOK.md`, `infra/runpod/*` |

**Mental model:** the POC changes *resources, data volume, and model size* — never the code paths.
Every service is injected through the same interface, so scaling up is a config/endpoint change.